# Fine-tune Qwen2.5-7B-Instruct (QLoRA) on the citation task

`scripts/finetune_llm.py`: the model gets the same prompt as the zero-shot run (citing sentence +
our reranker's top-20 evidence, answer A/B/C) and is trained on the 2141 training citations to
put its probability on the right letter. 4-bit weights, LoRA adapters (about 0.5% of the
parameters are trained), 1 epoch. Then dev and test are predicted.

Why: zero-shot, this model was weaker than our trained models (dev macro-F1 0.394), and perfect
evidence does not help BiomedBERT, so the limit is how well the model reads the evidence. This
gives that job to a 7B model trained on the task.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow
Google Drive access. Expect roughly **3 to 4 hours** on a T4; the training cell prints an ETA every
10 steps. Progress is saved to Google Drive every 25 steps: **if Colab disconnects, Run all again
and it continues** from the last save. To go about twice as fast with a smaller model, change
`MODEL` below to `Qwen/Qwen2.5-3B-Instruct` before running.

For reference: our current best (MultiVerS + 9-label BiomedBERT) has dev macro-F1 0.726, and on
test accuracy 0.642 / macro-F1 0.538; the paper system 0.584 / 0.524.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
TAG = MODEL.split('/')[-1].lower()
RES = f'/content/drive/MyDrive/CiteAudit_runs/llm_ft/{TAG}'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.45" accelerate bitsandbytes peft

## Train, then predict dev and test

In [ ]:
!python scripts/finetune_llm.py --model {MODEL} --load-4bit \
    --train data_train/target/ours_20_train_cv --dev data_train/target/ours_20 \
    --predict test=data_train/target/ours_20_test \
    --state-dir {RES}/state --pred-dir {RES}/pred 2>&1 | grep -vE "Warning|warn\(|Loading"

## Score

In [ ]:
!python scripts/score_multivers.py {RES}/pred/dev.jsonl --split dev
!python scripts/score_multivers.py {RES}/pred/ours_20_test.jsonl --split test

## Download

Zips the predictions and training log (the 200 MB training state stays in Drive). Put the unzipped
folder in the repository root and tell Claude.

In [ ]:
!cd {RES} && zip -qr /content/{TAG}_ft.zip pred
from google.colab import files
files.download(f'/content/{TAG}_ft.zip')